In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

G = 0.5
def simulate_one(mass, x, y, vx, vy, dt=0.2, max_steps=2000):
    horizon = np.sqrt(mass) * 0.8 + 8
    px, py = x, y
    min_dist = np.hypot(px, py)
    for step in range(max_steps):
        r = np.hypot(px, py)
        if r < horizon:
            return "SWALLOWED", step*dt, min_dist
        if r > 2000: # escaped
            return "ESCAPE", step*dt, min_dist
        # gravity
        a = G * mass / (r**3)
        vx -= px * a * dt
        vy -= py * a * dt
        px += vx * dt
        py += vy * dt
        min_dist = min(min_dist, r)
    # check stability - if still orbiting after max_steps
    return "STABLE", max_steps*dt, min_dist

rows = []
for _ in tqdm(range(10000)):
    mass = np.random.uniform(500, 8000)
    ang = np.random.uniform(0, 2*np.pi)
    r = np.random.uniform(40, 600)
    x = np.cos(ang) * r
    y = np.sin(ang) * r
    # orbital speed with noise
    v_orb = np.sqrt(G * mass / r) if r>0 else 0
    v_scale = np.random.uniform(0.2, 2.0)
    v_ang = ang + np.pi/2 + np.random.uniform(-0.5, 0.5)
    vx = np.cos(v_ang) * v_orb * v_scale
    vy = np.sin(v_ang) * v_orb * v_scale

    fate, t_swallow, min_d = simulate_one(mass, x, y, vx, vy)

    # features
    rows.append({
        "bh_mass": mass,
        "dist": r,
        "vx": vx, "vy": vy,
        "speed": np.hypot(vx, vy),
        "v_ratio": v_scale, # speed / orbital speed
        "ang_mom": x*vy - y*vx,
        "energy": 0.5*(vx**2+vy**2) - G*mass/r,
        "fate": fate,
        "time_to_fate": t_swallow,
        "min_dist": min_d
    })

df = pd.DataFrame(rows)
df.to_csv("blackhole_orbits.csv", index=False)
print(df['fate'].value_counts())
print(df.head())

100%|██████████| 10000/10000 [00:54<00:00, 182.17it/s]


fate
STABLE       7143
SWALLOWED    2526
ESCAPE        331
Name: count, dtype: int64
       bh_mass        dist        vx        vy     speed   v_ratio  \
0  7776.166643  239.034814  0.958612 -7.998997  8.056233  1.997537   
1  4406.688081  468.382381 -0.569132  3.822052  3.864194  1.781632   
2  2777.589065  532.326388  0.235717 -0.434205  0.494061  0.305880   
3  3145.915744  507.702568  3.257473 -0.480507  3.292722  1.870687   
4  6711.389689  482.349857 -0.389654 -0.753430  0.848226  0.321589   

       ang_mom     energy       fate  time_to_fate    min_dist  
0  1922.080969  16.185685     ESCAPE         317.8  238.415899  
1  1599.467482   2.761842     STABLE         400.0  468.382381  
2   245.592978  -2.486867     STABLE         400.0  155.908718  
3  1595.967001   2.322820     STABLE         400.0  475.576254  
4   379.454699  -6.597230  SWALLOWED         188.2   74.123972  


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import pickle

df = pd.read_csv("blackhole_orbits.csv")

# Features that user will have in real-time
FEATURES = ["bh_mass","dist","speed","v_ratio","ang_mom","energy"]
X = df[FEATURES]
y = df["fate"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, n_jobs=-1, eval_metric='mlogloss')
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred):.4f}\n")
print(classification_report(y_test, pred))

# save
with open("orbit_model.pkl","wb") as f:
    pickle.dump(model, f)

print("Saved orbit_model.pkl")

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got ['ESCAPE' 'STABLE' 'SWALLOWED']

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import pickle

df = pd.read_csv("blackhole_orbits.csv")
FEATURES = ["bh_mass","dist","speed","v_ratio","ang_mom","energy"]
X = df[FEATURES]
y = df["fate"]

le = LabelEncoder()
y_enc = le.fit_transform(y) # ESCAPE, STABLE, SWALLOWED -> 0,1,2

X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, n_jobs=-1)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
print(classification_report(y_test, pred, target_names=le.classes_))

with open("orbit_model.pkl","wb") as f:
    pickle.dump(model, f)
with open("label_encoder.pkl","wb") as f:
    pickle.dump(le, f)

print("Saved model + encoder")

Accuracy: 0.9755
              precision    recall  f1-score   support

      ESCAPE       0.90      0.82      0.86        66
      STABLE       0.98      0.99      0.98      1429
   SWALLOWED       0.97      0.96      0.97       505

    accuracy                           0.98      2000
   macro avg       0.95      0.92      0.94      2000
weighted avg       0.98      0.98      0.98      2000

Saved model + encoder


In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import pickle

df = pd.read_csv("blackhole_orbits.csv")
FEATURES = ["bh_mass","dist","speed","v_ratio","ang_mom","energy"]

# Only train time for SWALLOWED cases (others = stable)
df_sw = df[df["fate"]=="SWALLOWED"].copy()
X = df_sw[FEATURES]
y = df_sw["time_to_fate"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, n_jobs=-1)
reg.fit(X_train, y_train)

pred = reg.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, pred):.2f} sec")

with open("time_model.pkl","wb") as f:
    pickle.dump(reg, f)

print("Saved time_model.pkl")

MAE: 12.17 sec
Saved time_model.pkl


In [ ]:
.

SyntaxError: invalid syntax (1933637684.py, line 1)